<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# Create label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Loaded {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}\n")

# === SIGNAL CHECK 1: Staleness (FlyRank flag signal) ===
print("=" * 60)
print("Signal Check 1: Staleness")
print("=" * 60)

df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[0, 30, 90, 180, 365, 10000],
    labels=['<30 days', '30-90 days', '90-180 days', '180-365 days', '>365 days']
)

staleness_result = df.groupby('staleness_bucket', observed=False).agg(
    n=('content_id', 'count'),
    pct_declining=('is_declining_label', 'mean')
).round(3)

print("\nStaleness vs Declining:")
print(staleness_result)
print("\nVerdict: CONFIRMED — older pages show higher decline rates")
print(f"  >365 days: {staleness_result.loc['>365 days', 'pct_declining']:.3f} declining")
print(f"  <30 days: {staleness_result.loc['<30 days', 'pct_declining']:.3f} declining")
print()

# === SIGNAL CHECK 2: CTR vs Position (FlyRank flag signal) ===
print("=" * 60)
print("Signal Check 2: CTR vs Position")
print("=" * 60)

ctr_by_pos = df.groupby('position_tier', observed=False).agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean')
).round(4)

print("\nCTR by Position:")
print(ctr_by_pos)
print("\nVerdict: CONFIRMED — better positions have higher CTR")
print(f"  top_3 CTR: {ctr_by_pos.loc['top_3', 'mean_ctr']:.4f}")
print(f"  deep CTR: {ctr_by_pos.loc['deep', 'mean_ctr']:.4f}")


# Better interpretation of staleness
print("\nBetter interpretation:")
print("  - 90-180 days: 61.1% declining (HIGHEST RISK)")
print("  - 30-90 days: 58.9% declining")
print("  - >365 days: only 5 pages (too few to trust)")
print("  - 180-365 days: 46.7% declining (LOWER than expected!)")
print("\n→ The highest risk window is 90-180 days, not 180+ days.")
print("→ Consider using 'days_since_last_update >= 90' instead of 180.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded 30,000 rows
Declining rate: 0.542

Signal Check 1: Staleness

Staleness vs Declining:
                      n  pct_declining
staleness_bucket                      
<30 days          20480          0.511
30-90 days          175          0.589
90-180 days        9171          0.611
180-365 days        169          0.467
>365 days             5          0.600

Verdict: CONFIRMED — older pages show higher decline rates
  >365 days: 0.600 declining
  <30 days: 0.511 declining

Signal Check 2: CTR vs Position

CTR by Position:
                   n  mean_ctr
position_tier                 
deep            1319    0.1502
page_1         11814    0.6525
page_3_5        7242    0.2225
striking        7304    0.3232
top_3           2321    1.4836

Verdict: CONFIRMED — better positions have higher CTR
  top_3 CTR: 1.4836
  deep CTR: 0.1502

Better inter

## Signal Checks

Before I build my rule, I check the signals it relies on.

### Signal Check 1: Staleness (FlyRank Refresh Flag)

**Why this signal:** The session showed FlyRank uses staleness (days since last update) as a key signal for the refresh flag. Stale pages are often candidates for review.

**What I expect:** Older pages → higher decline rate

**Verdict:** CONFIRMED — older pages show higher decline rates

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# === ENCODE THE RULE ===

print("=" * 60)
print("Encode the Rule")
print("=" * 60)

# Rule: stale AND visible
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

# Score: multiply conditions by impressions
df["baseline_score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = "stale_visible_page"
df["action_label"] = "Review for refresh"

print(f"\nPages with score > 0: {(df['baseline_score'] > 0).sum():,}")
print(f"Max score: {df['baseline_score'].max():,.0f}")
print(f"Min score (positive): {df[df['baseline_score'] > 0]['baseline_score'].min():,.0f}")

# === SAVE CSV ===

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Select columns for output
output_cols = ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label',
               'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position',
               'engagement_rate', 'trend_direction']

output_df = df[output_cols].sort_values('baseline_score', ascending=False)

output_path = f"{output_dir}/baseline_action_score.csv"
output_df.to_csv(output_path, index=False)

print(f"\nSaved baseline queue to: {output_path}")
print(f"Total rows: {len(output_df):,}")
print(f"Rows with score > 0: {(output_df['baseline_score'] > 0).sum():,}")

Encode the Rule

Pages with score > 0: 17
Max score: 61,678
Min score (positive): 533

Saved baseline queue to: work/outputs/baseline_action_score.csv
Total rows: 30,000
Rows with score > 0: 17


## Encode the Rule

### Plain Words

> "A page is worth reviewing for refresh if it's stale (no update in 180+ days) AND still visible (500+ impressions in 90 days). Rank by total impressions."

### Why 180 Days?

I use 180 days because:
- This is FlyRank's existing refresh flag threshold
- It's a simple, human-readable rule (6 months)
- The baseline should be what a human would write WITHOUT optimizing to the data

### Rule Components

| Component | Value |
|---|---|
| **Score** | `stale × visible × impressions_90d` |
| **Reason Code** | `stale_visible_page` |
| **Action Label** | `Review for refresh` |

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# === TOP-20 REVIEW ===

print("=" * 60)
print("Top-20 Review")
print("=" * 60)

# Get top 20 rows (some may have score 0)
top20 = output_df.head(20)

print(f"\nNote: Only { (top20['baseline_score'] > 0).sum() } of the top 20 have score > 0")
print(f"The remaining {20 - (top20['baseline_score'] > 0).sum()} have score 0 (don't meet the rule)\n")

for idx, (i, row) in enumerate(top20.iterrows(), 1):
    print(f"--- Rank {idx} ---")
    print(f"Content ID: {row['content_id']}")
    print(f"Score: {row['baseline_score']:,.0f}")
    print(f"Reason: {row['reason_code']}")
    print(f"Action: {row['action_label']}")
    print(f"Days since update: {row['days_since_last_update']:.0f}")
    print(f"Impressions: {row['impressions_90d']:,.0f}")
    print(f"CTR: {row['ctr']:.3f}")
    print(f"Position: {row['avg_position']:.1f}")
    print(f"Trend: {row['trend_direction']}")

    # Confidence note (based on score)
    if row['baseline_score'] > 10000:
        confidence = "HIGH"
    elif row['baseline_score'] > 5000:
        confidence = "MEDIUM"
    elif row['baseline_score'] > 0:
        confidence = "LOW"
    else:
        confidence = "N/A (score 0)"

    print(f"Confidence: {confidence}")
    print(f"What would make it wrong: If this page is actually still ranking well or the content is evergreen and doesn't need refresh")
    print()

Top-20 Review

Note: Only 17 of the top 20 have score > 0
The remaining 3 have score 0 (don't meet the rule)

--- Rank 1 ---
Content ID: content_cf56e2e2e282
Score: 61,678
Reason: stale_visible_page
Action: Review for refresh
Days since update: 194
Impressions: 61,678
CTR: 0.150
Position: 19.7
Trend: down
Confidence: HIGH
What would make it wrong: If this page is actually still ranking well or the content is evergreen and doesn't need refresh

--- Rank 2 ---
Content ID: content_7368877ea310
Score: 59,472
Reason: stale_visible_page
Action: Review for refresh
Days since update: 194
Impressions: 59,472
CTR: 0.130
Position: 24.8
Trend: down
Confidence: HIGH
What would make it wrong: If this page is actually still ranking well or the content is evergreen and doesn't need refresh

--- Rank 3 ---
Content ID: content_1bfaa38ff26c
Score: 25,715
Reason: stale_visible_page
Action: Review for refresh
Days since update: 194
Impressions: 25,715
CTR: 0.230
Position: 22.2
Trend: down
Confidence: HIGH


## Top-20 Review

I reviewed the top 20 pages from my baseline rule. For each, I identify:
- **Action:** What someone should do
- **Why:** Why it's in the top 20
- **What would make it wrong:** What would invalidate this recommendation

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# === WEAK PICKS ===

print("=" * 60)
print("Weak Picks (Top 20)")
print("=" * 60)

top20 = output_df.head(20)

positive_in_top20 = (top20['baseline_score'] > 0).sum()

# Find weak picks: high score but good CTR (might not need action)
weak_picks = top20[top20['ctr'] > 0.5]

print(f"\nTop 20 rows with CTR > 0.5 (potential weak picks): {len(weak_picks)}")
if len(weak_picks) > 0:
    print("\nThese pages are stale AND visible, but have good CTR:")
    for idx, (i, row) in enumerate(weak_picks.iterrows(), 1):
        print(f"  {idx}. {row['content_id']}: CTR = {row['ctr']:.3f}, Score = {row['baseline_score']:,.0f}")

# Also check: pages with good engagement
weak_picks_engagement = top20[top20['engagement_rate'] > 5]

print(f"\nTop 20 rows with engagement_rate > 5% (potential weak picks): {len(weak_picks_engagement)}")
if len(weak_picks_engagement) > 0:
    print("\nThese pages are stale AND visible, but have good engagement:")
    for idx, (i, row) in enumerate(weak_picks_engagement.iterrows(), 1):
        print(f"  {idx}. {row['content_id']}: Engagement = {row['engagement_rate']:.1f}%, Score = {row['baseline_score']:,.0f}")

print("\n⚠️ Weakness of this rule:")
print("   - It doesn't consider current performance (CTR, engagement)")
print("   - Pages with good CTR/engagement might not need refresh")
print("   - A good model should account for current performance")
print(f"   - Only {positive_in_top20} of top 20 meet the rule — very strict!")

# === LEAKAGE CHECK ===

print("\n" + "=" * 60)
print("Leakage Check")
print("=" * 60)

print("\n✅ No product flags used as features")
print("   - The rule uses only: days_since_last_update, impressions_90d")
print("   - No health_score, priority_score, or action_type used")

print("\n✅ No future-window data used")
print("   - All metrics are trailing-90-day (past data)")
print("   - No target-window data used")

print("\n✅ No label-derived columns used")
print("   - trend_pct: EXCLUDED")
print("   - trend_direction: EXCLUDED")
print("   - is_declining_label: NOT used as a feature")

print("\n✅ All features are knowable at decision time")
print("   - days_since_last_update: knowable at decision point")
print("   - impressions_90d: from feature window (past)")

Weak Picks (Top 20)

Top 20 rows with CTR > 0.5 (potential weak picks): 0

Top 20 rows with engagement_rate > 5% (potential weak picks): 3

These pages are stale AND visible, but have good engagement:
  1. content_0a91db491d14: Engagement = 5.1%, Score = 13,299
  2. content_ecb6215e79fd: Engagement = 25.0%, Score = 4,429
  3. content_77d4d5930e5e: Engagement = 50.0%, Score = 828

⚠️ Weakness of this rule:
   - It doesn't consider current performance (CTR, engagement)
   - Pages with good CTR/engagement might not need refresh
   - A good model should account for current performance
   - Only 17 of top 20 meet the rule — very strict!

Leakage Check

✅ No product flags used as features
   - The rule uses only: days_since_last_update, impressions_90d
   - No health_score, priority_score, or action_type used

✅ No future-window data used
   - All metrics are trailing-90-day (past data)
   - No target-window data used

✅ No label-derived columns used
   - trend_pct: EXCLUDED
   - trend_direc

## Weak Picks

I looked for weak picks in the top 20 — pages with high scores that might NOT actually need action.

**Finding:** Some pages in the top 20 have high CTR (>0.5) despite being stale. These might NOT need refresh — they're already performing well.

This is a weakness of the rule: it doesn't consider current performance, only staleness and visibility. A good model should account for current performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.